In [ ]:
#%%
import pandas as pd
import numpy as np

# ------------------------- STEP 1: LOAD DATA -------------------------
#%%
# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")
# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")
# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")
# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")
# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")
# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')

#%%
# Define mapping for admission types
emergency_types = ['DIRECT EMER.', 'EW EMER.', 'URGENT']
normal_types = ['AMBULATORY OBSERVATION', 'DIRECT OBSERVATION', 'ELECTIVE', 
                'EU OBSERVATION', 'OBSERVATION ADMIT', 'SURGICAL SAME DAY ADMISSION']

# Create a new column 'admission_category' based on the mapping
admissions['admission_category'] = admissions['admission_type'].apply(
    lambda x: 'EMERGENCY' if x in emergency_types else 'NORMAL'
)

# Check if mapping worked correctly
print(admissions['admission_category'].value_counts())

In [ ]:
# Read codes
ccs_dx_mapping = pd.read_csv("/root/MIMICIV/src/codes/CCS_DX_mapping.csv")
ccs_dx_categories = pd.read_csv("/root/MIMICIV/src/codes/CCS_DX_categories.csv")
ccs_pcs_categories = pd.read_csv("/root/MIMICIV/src/codes/CCS_PCS_categories.csv")
ccs_pcs_mapping = pd.read_csv("/root/MIMICIV/src/codes/CCS_PCS_mapping.csv")

In [ ]:
len(ccs_dx_mapping['category_code'].unique())

In [ ]:
len(ccs_dx_categories)

In [ ]:
ccs_dx = pd.merge(ccs_dx_mapping, ccs_dx_categories, how='inner', on='category_code')
ccs_dx.head()

In [ ]:
ccs_dx['vocabulary_id'].unique()
ccs_dx['vocabulary_id'] = ccs_dx['vocabulary_id'].replace(
    {
    'ICD9CM': '9',
    'ICD10CM': '10'
    })
ccs_dx['key'] = ccs_dx['code'].astype(str) + '_' + ccs_dx['vocabulary_id']

In [ ]:
ccs_dx.head()

In [ ]:
ccs_dx_clean = ccs_dx.drop_duplicates(subset='key', keep='first')

In [ ]:
# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'diagnosis_description', 'icd_code', 'icd_version']]
procedures = procedures[['subject_id', 'hadm_id', 'procedure_description', 'icd_code', 'icd_version']]


In [ ]:
diagnoses.head()

In [ ]:
len(diagnoses)

In [ ]:
diagnoses['key'] = diagnoses['icd_code'].astype(str) + '_' + diagnoses['icd_version'].astype(str)

In [ ]:
diagnoses.head()

In [ ]:
len(diagnoses)

In [ ]:
diagnoses_ccs = diagnoses.merge(ccs_dx, how='left', on='key')

In [ ]:
diagnoses_ccs_clean = diagnoses.merge(ccs_dx_clean, how='left', on='key')

In [ ]:
diagnoses_ccs.head()

In [ ]:
diagnoses_ccs[pd.isna(diagnoses_ccs['vocabulary_id'])]

In [ ]:
diagnoses_ccs[pd.isna(diagnoses_ccs['icd_version'])]

In [ ]:
len(diagnoses_ccs)- len(diagnoses)

In [ ]:
len(diagnoses_ccs_clean) - len(diagnoses)

In [ ]:
ccs_dx[ccs_dx['key'].duplicated(keep=False)]

In [ ]:
ccs_dx[ccs_dx['key'] == '31200_9']

In [ ]:
ccs_dx_mapping[ccs_dx_mapping['code'] == '31200']

In [ ]:
ccs_dx_categories[ccs_dx_categories['category_code'] == 6521]

In [ ]:
diagnoses_ccs_clean.head()

In [ ]:
# If missing 'code', fill it with 'icd_code'
# This is to ensure that we have a code for each diagnosis, even if it doesn't map
diagnoses_ccs_clean['code'] = diagnoses_ccs_clean['code'].fillna(diagnoses_ccs_clean['icd_code'])
diagnoses_ccs_clean['category_code'] = diagnoses_ccs_clean['category_code'].fillna(diagnoses_ccs_clean['icd_code'])


In [ ]:
# If missing 'category_desc', fill it with 'diagnosis_description'
# This is to ensure that we have a description for each diagnosis, even if it doesn't map
diagnoses_ccs_clean['category_desc'] = diagnoses_ccs_clean['category_desc'].fillna(diagnoses_ccs_clean['diagnosis_description'])

In [ ]:
no_map_diagnoses = diagnoses_ccs_clean[pd.isna(diagnoses_ccs_clean['code_chapter'])][['icd_code', 'icd_version','category_desc']].drop_duplicates()
#pd.DataFrame(no_map_diagnoses, columns=['icd_code', 'icd_version','category_desc']).to_csv("/root/MIMICIV/src/codes/no_map_diagnoses.csv", index=False)

In [ ]:
no_map_diagnoses

In [ ]:
diagnoses_ccs_clean[pd.isna(diagnoses_ccs_clean['code_chapter'])].head()

In [ ]:
len(diagnoses_ccs_clean['category_code'].unique())

In [ ]:
len(diagnoses_ccs_clean['icd_code'].unique())

In [ ]:
# need to do the same for procedures
